In [ ]:
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import CLIPProcessor, CLIPModel

In [ ]:
ds = load_dataset("ILSVRC/imagenet-1k", split="train", streaming=True)
class_label = ds.features["label"]
model_name = "openai/clip-vit-base-patch16"
processor = CLIPProcessor.from_pretrained(model_name)
model = CLIPModel.from_pretrained(model_name)

In [ ]:
prompt_fmt = "a photo of {}"
text = [prompt_fmt.format(class_label.names[i]) for i in range(class_label.num_classes)]
text[:5]

In [ ]:
def zero_shot_demo(batch):
    images = batch["image"]
    labels = batch["label"]

    inputs = processor(
        text=text,
        images=images,
        return_tensors="pt",
        padding=True,
    )
    outputs = model(**inputs)
    # The image-text similarity score
    logits_per_image = outputs.logits_per_image
    # Take the softmax to get the label probabilities
    probs = logits_per_image.softmax(dim=1)
    top_values, top_indices = probs.topk(k=5, dim=1)

    fig, axes = plt.subplots(4, 2, figsize=(20, 40))
    axes = axes.flatten()
    for i in range(8):
        # [C, H, W] -> [H, W, C]
        image = images[i]
        label = class_label.int2str(labels[i])
        axes[i].imshow(image)
        axes[i].axis("off")

        # Create title with ground truth and top 5 predictions
        title = f"Ground Truth: {label}\n"
        title += "Top 5 Predictions:\n"
        for j in range(5):
            idx = top_indices[i][j].item()
            score = top_values[i][j].item()
            pred_label = class_label.int2str(idx)
            title += f"{pred_label} ({score:.4f})\n"

        axes[i].set_title(title)

In [ ]:
batched_ds = ds.batch(8)
for batch in batched_ds:
    zero_shot_demo(batch)
    break